# 11 · Thí nghiệm điểm mix trên last.fm, so với MPD (ticket #57)

Hai bộ dữ liệu khác bản chất: playlist sắp sẵn (MPD) và lịch sử nghe thật (last.fm, Music4All-Onion). Ba câu hỏi, cùng một cách đo trên cả hai bộ (`src/rerank/mix_datasets.py`, có test):

- **E1:** người nghe có nối bài mượt hơn ngẫu nhiên không? Cặp liền nhau thật so với cặp ngẫu nhiên (xáo bài kế tiếp trong cùng split val): % cặp mix được, ΔBPM, |Δenergy|, khoảng tin cậy 95% bootstrap theo phiên. Báo trên mọi phiên và khi bỏ phiên dài hơn p99.
- **E2:** điểm mix học (#23) huấn luyện lại trên cặp train của từng bộ: loss val, Recall@10 trong batch, đặt cạnh Recall@10 của điểm mix theo luật trên cùng các batch val. last.fm lấy mẫu phiên train cho số cặp ngang MPD.
- **E3:** Spearman giữa điểm học và luật trên cặp ngẫu nhiên (cùng cách đo với #24).

Chỉ học từ cặp train của mỗi bộ; không đem mô hình học trên MPD áp sang last.fm.

**Input** (Add Input): `titeohihi/mixnext-splited` (MPD) và `mixnext-lastfm` (notebook 10: `lastfm/tracks.parquet`, `lastfm/sessions.parquet`).

**Output** (`/kaggle/working/out/lastfm_mix/`): `summary.csv`, `results.json`, `e1_mixable.png`, `e2_recall.png`. Không đăng Dataset; hình dùng cho báo cáo (#29).

Settings: **Accelerator GPU** (học điểm mix hai lần, mỗi lần tối đa 30 phút), bật **Internet**.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #57 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
import json, time
from pathlib import Path
import pandas as pd

MPD = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent      # bộ MPD có eval_targets
LASTFM = sorted(Path("/kaggle/input").rglob("lastfm_log.json"))[0].parent        # bộ last.fm của notebook 10
OUT = Path("/kaggle/working/out/lastfm_mix")
OUT.mkdir(parents=True, exist_ok=True)
mpd_tracks, mpd_sessions = pd.read_parquet(MPD / "tracks.parquet"), pd.read_parquet(MPD / "sessions.parquet")
print("MPD |", len(mpd_tracks), "bài |", mpd_sessions["session_id"].nunique(), "phiên |", len(mpd_sessions), "lượt")
print("last.fm:", LASTFM)

In [ ]:
# E1–E3 trên MPD trước; rồi last.fm: giữ đủ phiên val, lấy mẫu phiên train cho số cặp ngang MPD ngay lúc đọc (vừa RAM)
import numpy as np
from src.rerank.mix_datasets import load_sessions_for_mix, run_dataset, summary_table
from src.rerank.mix_learned_train import MixTrainParams

params = MixTrainParams()   # như #23
results = {}
t0 = time.time()
results["MPD"] = run_dataset(mpd_tracks, mpd_sessions, params, random_seed=0)
print(f"MPD xong sau {time.time() - t0:.0f} giây")
target = results["MPD"]["e2"]["n_train_pairs"]
lastfm_sessions, sampled = load_sessions_for_mix(LASTFM / "sessions.parquet", target, np.random.default_rng(0))
print("last.fm sau lấy mẫu:", sampled, "| cặp train MPD:", f"{target:,}")
results["last.fm"] = run_dataset(pd.read_parquet(LASTFM / "tracks.parquet"), lastfm_sessions, params, random_seed=0,
                                 notes={"sampled_train": sampled, "mpd_train_pairs": target})
print(f"last.fm xong sau {time.time() - t0:.0f} giây")

table = summary_table(results)
table.to_csv(OUT / "summary.csv")
(OUT / "results.json").write_text(json.dumps({k: {e: v for e, v in r.items() if e != "encoders"} for k, r in results.items()}, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print(table.T.to_string())

In [ ]:
# Hình E1 (% cặp mix được, thật vs ngẫu nhiên, CI 95%) và E2 (Recall@10 trong batch: học, luật, ngẫu nhiên)
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
x = np.arange(len(results))
for k, (row, color) in enumerate([("thật", "#2a78d6"), ("ngẫu nhiên", "#eb6834")]):
    vals = [r["e1"]["all"]["table"][row]["mixable_pct"] for r in results.values()]
    err = [[v - r["e1"]["all"]["table"][row]["mixable_pct_low"] for v, r in zip(vals, results.values())],
           [r["e1"]["all"]["table"][row]["mixable_pct_high"] - v for v, r in zip(vals, results.values())]]
    bars = ax.bar(x + (k - 0.5) * 0.36, vals, width=0.34, yerr=err, capsize=4, color=color, label=f"cặp {row}")
    ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=9)
ax.set_xticks(x, list(results))
ax.set_ylabel("cặp mix được (%)")
ax.set_title("E1 · Cặp liền nhau thật so với ngẫu nhiên (split val, CI 95% theo phiên)", loc="left")
ax.legend(frameon=False)
fig.savefig(OUT / "e1_mixable.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
labels = [("learned", "điểm học"), ("rule", "luật")]
for k, (key, label) in enumerate(labels):
    vals = [r["e2"]["recall_at_10"][key] for r in results.values()]
    bars = ax.bar(x + (k - 1) * 0.26, vals, width=0.24, label=label)
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
rand = [r["e2"]["random_recall_at_10"] for r in results.values()]
bars = ax.bar(x + 0.26, rand, width=0.24, color="#b5b3ad", label="ngẫu nhiên")
ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
ax.set_xticks(x, list(results))
ax.set_ylabel("Recall@10 trong batch 1.024 cặp val")
ax.set_title("E2 · Hàm mix nào đoán đúng bài kế tiếp hơn?", loc="left")
ax.legend(frameon=False)
fig.savefig(OUT / "e2_recall.png", dpi=200)
plt.show()

In [ ]:
# E2: loss val và Recall@10 trong batch theo epoch (epoch 0 = bộ mã hóa chưa học)
for name, r in results.items():
    history = pd.DataFrame(r["e2"]["history"]).set_index("epoch")
    print(f"{name} (giữ epoch {r['e2']['best_epoch']})")
    print(history.round(4).to_string())
    print()

## Kết quả (chưa chạy)

Dán vào đây bảng `summary.csv` (chuyển vị), thời gian chạy, và nhận xét từ hai hình.

## Giới hạn (cho mục Tranh luận của báo cáo)

- **energy của last.fm là thang thay thế** (hạng phần trăm loudness, #56), không cùng nghĩa với energy của Spotify ở MPD; |Δenergy| hai bộ không so trực tiếp được.
- **Tông của last.fm tự ước lượng** từ HPCP (Krumhansl–Schmuckler, #56; tương quan 0,7 với độ mạnh tông của Essentia), nên "tông tương thích" và "cặp mix được" ở last.fm có thêm sai số đo.
- **Nghe cả album theo thứ tự**: "chuyển bài" ở last.fm phản ánh cả cách nghệ sĩ sắp xếp album, không chỉ lựa chọn của người nghe.
- **last.fm không ghi bài bị bỏ qua** (skip): cặp liền nhau là những bài đã được nghe đủ lâu để ghi lại, không biết người nghe có chủ động chọn không.
- **Phiên cắt theo khoảng nghỉ 30 phút** (#55) là một lựa chọn; số cặp thay đổi mạnh theo ngưỡng (68 / 106 / 133 triệu cặp với 15 / 30 / 60 phút).
- **Recall@10 của điểm học hơi lạc quan:** điểm học dừng sớm theo loss trên cặp val (#23), rồi được chấm Recall@10 cũng trên cặp val; luật không có bước chọn nào trên val.
- **Không đem mô hình học trên MPD áp sang last.fm** (và ngược lại): mỗi bộ có điểm mix học riêng, huấn luyện chỉ trên cặp train của bộ đó.